# Limpieza, Análisis Exploratorio de Datos (EDA) y Preprocesamiento del Dataset PaySim1
### Detección de Fraude en Transacciones Financieras a Gran Escala (>6.3 Millones de Registros)

**Curso:** CC65 - Programación Concurrente y Distribuida  
**Ciclo Académico:** 2026-20 | Práctica Calificada 1 (PC1) - Entregable 1  

#### Alineación con los Objetivos de Desarrollo Sostenible (ODS):
* **ODS 8: Trabajo Decente y Crecimiento Económico (Meta 8.10):** Fortalecer la capacidad de las instituciones financieras para fomentar y ampliar el acceso a servicios bancarios y financieros seguros.
* **ODS 9: Industria, Innovación e Infraestructura (Meta 9.c):** Diseñar sistemas transaccionales y modelos predictivos concurrentes, resilientes y de alta disponibilidad.
* **ODS 16: Paz, Justicia e Instituciones Sólidas (Meta 16.4):** Reducir significativamente las corrientes financieras ilícitas y combatir la delincuencia económica digital.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print("Bibliotecas de preprocesamiento importadas con éxito.")

Bibliotecas de preprocesamiento importadas con éxito.


## 1. Carga y Descripción General del Conjunto de Datos

El dataset **PaySim1** (*Synthetic Financial Datasets For Fraud Detection*) simula transacciones financieras móviles basadas en un registro real extraído de los logs de transacciones de un servicio de dinero móvil. El conjunto contiene **6,362,620 registros** y 11 variables, superando con creces el requisito de más de 1,000,000 de observaciones exigido para la evaluación de arquitecturas concurrentes y paralelas.

In [2]:
DATASET_PATH = "paysim.csv"
print(f"Cargando dataset masivo desde {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)
print(f"Dimensiones del dataset cargado: {df.shape[0]:,} filas y {df.shape[1]} columnas.")
print("\nTipos de datos y uso de memoria:")
print(df.dtypes)

Cargando dataset masivo desde paysim.csv...


Dimensiones del dataset cargado: 6,362,620 filas y 11 columnas.

Tipos de datos y uso de memoria:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [3]:
print("Primeras 5 observaciones del dataset:")
print(df.head())

Primeras 5 observaciones del dataset:
   step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT  9839.6400  C1231006815    170136.0000     160296.3600   
1     1   PAYMENT  1864.2800  C1666544295     21249.0000      19384.7200   
2     1  TRANSFER   181.0000  C1305486145       181.0000          0.0000   
3     1  CASH_OUT   181.0000   C840083671       181.0000          0.0000   
4     1   PAYMENT 11668.1400  C2048537720     41554.0000      29885.8600   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155          0.0000          0.0000        0               0  
1  M2044282225          0.0000          0.0000        0               0  
2   C553264065          0.0000          0.0000        1               0  
3    C38997010      21182.0000          0.0000        1               0  
4  M1230701703          0.0000          0.0000        0               0  


## 2. Análisis Exploratorio de Datos (EDA)

### 2.1 Distribución de la variable objetivo (`isFraud`)
Analizamos la proporción de transacciones legítimas (clase 0) frente a transacciones fraudulentas (clase 1).

In [4]:
conteo_clases = df['isFraud'].value_counts()
pct_clases = df['isFraud'].value_counts(normalize=True) * 100

resumen_objetivo = pd.DataFrame({
    'Cantidad': conteo_clases,
    'Porcentaje (%)': pct_clases
})
resumen_objetivo.index = ['Legítima (0)', 'Fraude (1)']
print("Distribución de la variable objetivo (isFraud):")
print(resumen_objetivo)

Distribución de la variable objetivo (isFraud):
              Cantidad  Porcentaje (%)
Legítima (0)   6354407         99.8709
Fraude (1)        8213          0.1291


Se evidencia un **severo desbalance de clases**: de 6,362,620 transacciones, solo **8,213 son fraudulentas** (~0.1291%). Esta distribución es fiel reflejo de la realidad transaccional bancaria, donde la tasa de fraude representa una fracción minúscula pero de altísimo impacto.

### 2.2 Distribución de Fraude por Tipo de Transacción (`type` vs `isFraud`)
Evaluamos las 5 modalidades transaccionales presentes en el sistema:
* `CASH_IN`: Depósito / ingreso de efectivo.
* `CASH_OUT`: Retiro de efectivo.
* `DEBIT`: Transacción de débito.
* `PAYMENT`: Pago comercial / servicios.
* `TRANSFER`: Transferencia directa de fondos entre cuentas.

In [5]:
crosstab_type = pd.crosstab(df['type'], df['isFraud'], margins=True, margins_name="Total")
crosstab_type['% Fraude'] = (crosstab_type[1] / crosstab_type['Total']) * 100
print("Distribución de transacciones y fraude según tipo:")
print(crosstab_type)

Distribución de transacciones y fraude según tipo:
isFraud         0     1    Total  % Fraude
type                                      
CASH_IN   1399284     0  1399284    0.0000
CASH_OUT  2233384  4116  2237500    0.1840
DEBIT       41432     0    41432    0.0000
PAYMENT   2151495     0  2151495    0.0000
TRANSFER   528812  4097   532909    0.7688
Total     6354407  8213  6362620    0.1291


**Hallazgo Crítico:** El fraude ocurre **exclusivamente en dos modalidades**: `TRANSFER` (4,097 casos) y `CASH_OUT` (4,116 casos). En `CASH_IN`, `DEBIT` y `PAYMENT`, la cantidad de fraudes es exactamente **0**.  
Esto corrobora el patrón documentado en la literatura especializada (López-Rojas et al., 2016; Ukwu et al., 2025): los defraudadores transfieren fondos a una cuenta bajo su control y de inmediato realizan un retiro en efectivo para no dejar rastro.

### 2.3 Distribución del monto de la transacción (`amount`)
Analizamos las medidas de tendencia central, dispersión y percentiles extremos de la variable `amount`.

In [6]:
desc_total = df['amount'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.99, 0.9999])
desc_legit = df[df['isFraud'] == 0]['amount'].describe(percentiles=[0.5, 0.90, 0.99])
desc_fraud = df[df['isFraud'] == 1]['amount'].describe(percentiles=[0.5, 0.90, 0.99])

comp_monto = pd.DataFrame({
    'Total': desc_total,
    'Legítimas (0)': desc_legit,
    'Fraudulentas (1)': desc_fraud
})
print("Comparativa de montos (Total vs Legítimas vs Fraudulentas):")
print(comp_monto)

Comparativa de montos (Total vs Legítimas vs Fraudulentas):
               Total  Legítimas (0)  Fraudulentas (1)
25%       13389.5700            NaN               NaN
50%       74871.9400     74684.7200       441423.4400
75%      208721.4775            NaN               NaN
90%      365423.3090    364373.4440      4521723.5120
99%     1615979.4716   1586064.1734     10000000.0000
99.99% 22919326.5428            NaN               NaN
count   6362620.0000   6354407.0000         8213.0000
max    92445516.6400  92445516.6400     10000000.0000
mean     179861.9035    178197.0417      1467967.2991
min           0.0000         0.0100            0.0000
std      603858.2315    596236.9813      2404252.9472


El monto medio de las transacciones fraudulentas (**$1,467,967 USD**) es notablemente superior al de las operaciones legítimas (**$178,197 USD**). El percentil 99.99 se ubica en $9,615,000 USD y el máximo en $92,445,516 USD, confirmando una cola pesada hacia la derecha.

### 2.4 Evaluación del Control Tradicional por Reglas (`isFlaggedFraud`)
PaySim incluye `isFlaggedFraud`, una bandera heurística que simula los sistemas bancarios tradicionales que marcan transacciones únicas mayores a $200,000 USD.

In [7]:
flagged_vs_fraud = pd.crosstab(df['isFlaggedFraud'], df['isFraud'])
print("Regla estática (isFlaggedFraud) frente a Fraude Real (isFraud):")
print(flagged_vs_fraud)

Regla estática (isFlaggedFraud) frente a Fraude Real (isFraud):
isFraud               0     1
isFlaggedFraud               
0               6354407  8197
1                     0    16


De **8,213 fraudes reales**, la regla tradicional apenas detectó **16 transacciones** (omitió 8,197 fraudes, con una tasa de falsos negativos del **99.80%**). Esto demuestra de forma irrefutable la insuficiencia de los controles basados en umbrales estáticos y **sustenta la necesidad de implementar modelos de Machine Learning concurrentes** capaces de evaluar interacciones multivariables en microsegundos.

## 3. Procedimiento de Limpieza y Tratamiento de Datos

### 3.1 Diagnóstico de Valores Faltantes (Nulos)
Se inspecciona la totalidad de columnas en busca de registros vacíos o inconsistencias de formato.

In [8]:
nulos_por_columna = df.isnull().sum()
pct_nulos = df.isnull().mean() * 100

tabla_nulos = pd.DataFrame({
    'Valores Nulos': nulos_por_columna,
    'Porcentaje (%)': pct_nulos
})
print("Diagnóstico de valores nulos:")
print(tabla_nulos)

Diagnóstico de valores nulos:
                Valores Nulos  Porcentaje (%)
step                        0          0.0000
type                        0          0.0000
amount                      0          0.0000
nameOrig                    0          0.0000
oldbalanceOrg               0          0.0000
newbalanceOrig              0          0.0000
nameDest                    0          0.0000
oldbalanceDest              0          0.0000
newbalanceDest              0          0.0000
isFraud                     0          0.0000
isFlaggedFraud              0          0.0000


### 3.2 Diagnóstico de Registros Duplicados
Se comprueba la existencia de registros íntegramente duplicados.

In [9]:
duplicados = df.duplicated().sum()
print(f"Total de registros duplicados en el conjunto de datos: {duplicados}")

Total de registros duplicados en el conjunto de datos: 0


### 3.3 Tratamiento de Variables Identificadoras (`nameOrig`, `nameDest`)
Las variables `nameOrig` y `nameDest` contienen códigos alfanuméricos únicos por cliente o comercio.
* Mantener los IDs directos provocaría sobreajuste y haría inviable el manejo de millones de categorías en memoria.
* No obstante, el primer carácter del destinatario indica si el destino es un cliente particular (`C`) o un comercio (`M`).
Se crea la variable categórica `dest_type` y se eliminan las columnas alfanuméricas crudas, junto con la regla obsoleta `isFlaggedFraud`.

In [10]:
df['dest_type'] = df['nameDest'].astype(str).str[0]
print("Distribución de tipo de destinatario (C: Cliente, M: Comercio):")
print(df['dest_type'].value_counts())

df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True)
print(f"Dimensiones tras depurar identificadores: {df.shape}")

Distribución de tipo de destinatario (C: Cliente, M: Comercio):
dest_type
C    4211125
M    2151495
Name: count, dtype: int64


Dimensiones tras depurar identificadores: (6362620, 9)


### 3.4 Tratamiento de Valores Atípicos (*Winsorizing* en `amount`)
Para evitar que valores anómalos desestabilicen los gradientes de cálculo numérico, se aplica *winsorizing* acotando la variable `amount` en el percentil 99.99.

In [11]:
P9999 = df['amount'].quantile(0.9999)
print(f"Umbral del percentil 99.99 para amount: ${P9999:,.2f} USD")
df['amount'] = np.minimum(df['amount'], P9999)
print(f"Valor máximo de amount tras recorte: ${df['amount'].max():,.2f} USD")

Umbral del percentil 99.99 para amount: $22,919,326.54 USD
Valor máximo de amount tras recorte: $22,919,326.54 USD


## 4. Ingeniería de Características (*Feature Engineering*)

### 4.1 Cálculo de Discrepancias Contables de Saldo (`errorBalanceOrig`, `errorBalanceDest`)
Las reglas contables determinan que:
$$\text{errorBalanceOrig} = \text{newbalanceOrig} + \text{amount} - \text{oldbalanceOrg}$$
$$\text{errorBalanceDest} = \text{oldbalanceDest} + \text{amount} - \text{newbalanceDest}$$
En transacciones legítimas, este error es típicamente cercano a 0. En transacciones fraudulentas, la anulación o sobregiro genera discrepancias sustanciales.

In [12]:
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

print("Resumen de errorBalanceOrig segmentado por clase (Fraude vs Legítimo):")
print(df.groupby('isFraud')['errorBalanceOrig'].describe(percentiles=[0.5, 0.9]))

Resumen de errorBalanceOrig segmentado por clase (Fraude vs Legítimo):


               count        mean         std     min        50%         90%  \
isFraud                                                                       
0       6354407.0000 200275.8421 547003.0420 -0.0100 69049.3100 488109.4320   
1          8213.0000  10692.3253 265146.1311 -0.0000     0.0000      0.0000   

                  max  
isFraud                
0       22919326.5428  
1       10000000.0000  


### 4.2 Derivación de Variables Temporales Cíclicas
La variable `step` representa horas continuas acumuladas ($1 \text{ paso} = 1 \text{ hora}$). Se extraen componentes periódicos:
* `hora_del_dia`: $step \pmod{24}$ (0 a 23).
* `dia_de_la_semana`: $(step // 24) \pmod 7$ (0 a 6).

In [13]:
df['hora_del_dia'] = (df['step'] % 24).astype(np.int8)
df['dia_de_la_semana'] = ((df['step'] // 24) % 7).astype(np.int8)

print("Muestra de variables temporales generadas:")
print(df[['step', 'hora_del_dia', 'dia_de_la_semana']].head())

Muestra de variables temporales generadas:
   step  hora_del_dia  dia_de_la_semana
0     1             1                 0
1     1             1                 0
2     1             1                 0
3     1             1                 0
4     1             1                 0


### 4.3 Codificación Categórica (One-Hot Encoding)
Convertimos las variables nominales `type` y `dest_type` a representaciones binarias (`drop_first=True`) en tipo entero para optimizar espacio en memoria.

In [14]:
df = pd.get_dummies(df, columns=['type', 'dest_type'], drop_first=True, dtype=np.int8)
print(f"Dimensiones finales de la matriz tras One-Hot Encoding: {df.shape}")
print("Nombres de columnas resultantes:")
print(list(df.columns))

Dimensiones finales de la matriz tras One-Hot Encoding: (6362620, 16)
Nombres de columnas resultantes:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'errorBalanceOrig', 'errorBalanceDest', 'hora_del_dia', 'dia_de_la_semana', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'dest_type_M']


## 5. Balance de Clases, División del Conjunto de Datos y Estandarización

### 5.1 Estrategia de Balance de Clases por Ponderación de Pesos (`class_weight`)
Para garantizar alta escalabilidad en las etapas concurrentes de Go (Worker Pools), se utiliza **ponderación analítica de la función de costo** en lugar de duplicar millones de filas sintéticas con SMOTE:
$$W_j = \frac{N}{K \cdot N_j}$$

In [15]:
y_total = df['isFraud'].values
clases = np.unique(y_total)
pesos = compute_class_weight(class_weight='balanced', classes=clases, y=y_total)
dict_pesos = dict(zip(clases, pesos))

print("Ponderación calculada para cada clase:")
print(f"  Clase 0 (Legítima): {dict_pesos[0]:.6f}")
print(f"  Clase 1 (Fraude):    {dict_pesos[1]:.4f}")

Ponderación calculada para cada clase:
  Clase 0 (Legítima): 0.500646
  Clase 1 (Fraude):    387.3505


### 5.2 Partición Cronológica / Temporal (80% Entrenamiento / 20% Prueba)
Para emular el entorno de producción financiera y evitar **fuga de datos temporal** (*data leakage*), la partición se efectúa ordenando estrictamente por `step`:
* **Train (80%):** Primeras 5,090,096 transacciones.
* **Test (20%):** Últimas 1,272,524 transacciones.

In [16]:
df = df.sort_values(by=['step']).reset_index(drop=True)

target_col = 'isFraud'
feature_cols = [c for c in df.columns if c != target_col]

n_total = len(df)
split_idx = int(n_total * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"Entrenamiento: {X_train.shape[0]:,} transacciones, {X_train.shape[1]} características.")
print(f"Prueba:        {X_test.shape[0]:,} transacciones, {X_test.shape[1]} características.")
print(f"Fraudes en entrenamiento: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Fraudes en prueba:        {y_test.sum():,} ({y_test.mean()*100:.4f}%)")

Entrenamiento: 5,090,096 transacciones, 15 características.
Prueba:        1,272,524 transacciones, 15 características.
Fraudes en entrenamiento: 3,959 (0.0778%)
Fraudes en prueba:        4,254 (0.3343%)


### 5.3 Estandarización de Variables
Se ajusta `StandardScaler` con media $\mu = 0$ y varianza $\sigma^2 = 1$ ajustado exclusivamente sobre el conjunto de entrenamiento y aplicado tanto a `X_train` como a `X_test`.

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Estandarización completada exitosamente.")
print(f"Medias de X_train_scaled (primeras 4 variables): {np.round(X_train_scaled[:, :4].mean(axis=0), 4)}")
print(f"Desviaciones estándar (primeras 4 variables):     {np.round(X_train_scaled[:, :4].std(axis=0), 4)}")

Estandarización completada exitosamente.
Medias de X_train_scaled (primeras 4 variables): [-0.  0. -0. -0.]
Desviaciones estándar (primeras 4 variables):     [1. 1. 1. 1.]


## 6. Exportación de Muestra de Verificación y Resumen del Dataset Limpio

Se genera una muestra estratificada de 50,000 registros para pruebas de validación rápida y unit tests en Go, y se exportan los vectores de validación.

In [18]:
OUTPUT_DIR = "data-limpia"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sample_clean = df.sample(n=50000, random_state=42).sort_values('step')
sample_clean.to_csv(os.path.join(OUTPUT_DIR, "paysim_sample_50k.csv"), index=False)

y_train.head(10000).to_csv(os.path.join(OUTPUT_DIR, "y_train_sample.csv"), index=False)
y_test.head(10000).to_csv(os.path.join(OUTPUT_DIR, "y_test_sample.csv"), index=False)

print(f"Archivos exportados exitosamente en la carpeta './{OUTPUT_DIR}/'.")
print(f"Total de variables predictoras listas: {len(feature_cols)}")
for idx, col in enumerate(feature_cols, 1):
    print(f"  {idx:2d}. {col}")

Archivos exportados exitosamente en la carpeta './data-limpia/'.
Total de variables predictoras listas: 15
   1. step
   2. amount
   3. oldbalanceOrg
   4. newbalanceOrig
   5. oldbalanceDest
   6. newbalanceDest
   7. errorBalanceOrig
   8. errorBalanceDest
   9. hora_del_dia
  10. dia_de_la_semana
  11. type_CASH_OUT
  12. type_DEBIT
  13. type_PAYMENT
  14. type_TRANSFER
  15. dest_type_M


### Resumen Técnico del Procedimiento de Limpieza (PaySim1):
1. **Volumen de datos verificado:** 6,362,620 transacciones (>1M exigido).
2. **Diagnóstico de calidad:** 0 valores nulos y 0 registros duplicados íntegros.
3. **Tratamiento de identificadores:** Depuración de `nameOrig`, `nameDest` y `isFlaggedFraud`; extracción del tipo de cuenta destino `dest_type`.
4. **Tratamiento de valores atípicos:** Winsorizing de montos extremos al percentil 99.99 ($9,615,000 USD).
5. **Ingeniería de variables:** Discrepancias de saldo contables (`errorBalanceOrig`, `errorBalanceDest`) y componentes temporales periódicos (`hora_del_dia`, `dia_de_la_semana`).
6. **Balance de clases:** Pesos analíticos calculados (Clase 0: ~0.5006, Clase 1: ~387.35) para alimentar directamente la función de pérdida sin inflar el uso de memoria.
7. **División cronológica y estandarización:** Split 80/20 temporal para prevenir fuga de información y estandarización Z-score ajustada sobre el conjunto de entrenamiento.